# NOAA AIS Data Extraction



In [ ]:
import pandas as pd
import requests
import io
import os
import warnings
from google.colab import drive

!pip install zstandard -q

#  MOUNT GOOGLE DRIVE
print("Mounting Google Drive...")
drive.mount('/content/drive')

# CONFIGURATION
output_folder = "/content/drive/MyDrive/LCO2 Transport Project Final"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

master_file = os.path.join(output_folder, "AIS_2025_Tanker.csv")
progress_file = os.path.join(output_folder, "AIS_2025_extraction_progress.txt")

# 2025 DATA SOURCE -  https://noaaocm.blob.core.windows.net/ais/csv2/csv2025/ais-2025-MM-DD.csv.zst
base_url = "https://noaaocm.blob.core.windows.net/ais/csv2/csv2025/ais-2025-{:02d}-{:02d}.csv.zst"

warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

cols_to_load = [
    'mmsi', 'base_date_time', 'imo', 'longitude', 'latitude', 'sog', 'cog',
    'heading', 'vessel_type', 'status', 'length', 'width', 'draft',
    'cargo', 'transceiver'
]

rename_map = {
    'mmsi': 'MMSI', 'base_date_time': 'BaseDateTime', 'imo': 'IMO',
    'longitude': 'LON', 'latitude': 'LAT', 'sog': 'SOG', 'cog': 'COG',
    'heading': 'Heading', 'vessel_type': 'VesselType', 'status': 'Status',
    'length': 'Length', 'width': 'Width', 'draft': 'Draft', 'cargo': 'Cargo',
    'transceiver': 'TransceiverClass'
}

processed_dates = set()
if os.path.exists(progress_file):
    with open(progress_file) as f:
        processed_dates = set(line.strip() for line in f if line.strip())
    print(f"Resuming: {len(processed_dates)} days already processed.")

print(f"Starting Extraction. Saving to: {master_file}")

#  LOOP OVER EVERY DAY OF YEAR
for month in range(1, 13):
    for day in range(1, 32):
        try:
            pd.Timestamp(2025, month, day)
        except ValueError:
            continue  # 2025 has 365 days

        date_str = f"2025-{month:02d}-{day:02d}"
        if date_str in processed_dates:
            print(f"Skipping as already processed: {date_str} ")
            continue

        url = base_url.format(month, day)
        print(f" Processing: {date_str} ")

        try:
            #  DOWNLOAD
            r = requests.get(url, timeout=180)
            if r.status_code != 200:
                print("Skipped - Not Found")
                continue

            #  LOAD AND PROCESS (zstd-compressed CSV directly, no ZIP wrapper)
            df = pd.read_csv(io.BytesIO(r.content), compression='zstd', usecols=cols_to_load)
            df.rename(columns=rename_map, inplace=True)
            print(f" Data Loaded: {len(df)} raw rows")

            # CLEANING
            df = df[df['TransceiverClass'].astype(str).str.strip().str.upper() == 'A']
            if 'VesselType' in df.columns:
                df['Cargo'] = df['Cargo'].fillna(df['VesselType'])
            mask_tanker = (df['Cargo'] >= 80) & (df['Cargo'] <= 89)
            mask_moving = (df['SOG'] >= 0.0) & (df['SOG'] <= 40.0)
            mask_physics = (
                (df['Length'] > 30) &
                (df['Width'] > 0) &
                (df['Draft'] > 0.5) &
                (df['LON'] < 0)
            )
            df = df[mask_tanker & mask_moving & mask_physics]

            df['BaseDateTime'] = pd.to_datetime(df['BaseDateTime'])
            hour_bucket = df['BaseDateTime'].dt.floor('h')
            df = df.loc[~pd.concat([df['MMSI'], hour_bucket], axis=1).duplicated()]

            # DROP COLUMNS
            cols_to_drop = ['VesselType', 'TransceiverClass']
            df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

            # SAVE
            use_header = not os.path.exists(master_file)
            df.to_csv(master_file, mode='a', index=False, header=use_header)

            print(f" -> Saved to Master: +{len(df)} rows")

            # MARK PROGRESS (so a re-run can skip this day)
            with open(progress_file, 'a') as f:
                f.write(date_str + "\n")

            # CLEAR FROM MEMORY
            del r, df
            print(" -> Parent data cleared from memory.")

        except Exception as e:
            print(f"Error: {e}")

print("COMPLETED EXTRACTING")


Mounting Google Drive...
Mounted at /content/drive
Starting Extraction. Saving to: /content/drive/MyDrive/LCO2 Transport Project Final/AIS_2025_Tanker.csv
 Processing: 2025-01-01 
 Data Loaded: 7337208 raw rows
 -> Saved to Master: +11249 rows
 -> Parent data cleared from memory.
 Processing: 2025-01-02 
 Data Loaded: 6684195 raw rows
 -> Saved to Master: +11382 rows
 -> Parent data cleared from memory.
 Processing: 2025-01-03 
 Data Loaded: 6585461 raw rows
 -> Saved to Master: +10933 rows
 -> Parent data cleared from memory.
 Processing: 2025-01-04 
 Data Loaded: 6844921 raw rows
 -> Saved to Master: +10950 rows
 -> Parent data cleared from memory.
 Processing: 2025-01-05 
 Data Loaded: 7608241 raw rows
 -> Saved to Master: +10713 rows
 -> Parent data cleared from memory.
 Processing: 2025-01-06 
 Data Loaded: 6453187 raw rows
 -> Saved to Master: +10665 rows
 -> Parent data cleared from memory.
 Processing: 2025-01-07 
 Data Loaded: 6445411 raw rows
 -> Saved to Master: +10850 rows
